# World Cup Model Sweep and Neural Network Experiments

This notebook tries a wider set of algorithms on the same frozen 2010-2022 FIFA World Cup audit: regularized logistic regression, random forest, extra trees, gradient boosting, histogram gradient boosting, XGBoost, shallow MLP, deeper MLP, and validation-weighted ensembles.

The repository does not include PyTorch or TensorFlow, so the neural-network experiments use scikit-learn's `MLPClassifier` with hidden-layer variants. No Python source files are created or modified.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'matches_with_elo.csv'
REPORTS = ROOT / 'reports'
YEARS = [2010, 2014, 2018, 2022]
SEED = 20260707
LABELS = {'H': 0, 'D': 1, 'A': 2}
ORDERED = np.array(['H', 'D', 'A'])

matches = pd.read_csv(DATA, parse_dates=['date']).sort_values(['date', 'match_id']).reset_index(drop=True)
assert matches.date.max() >= pd.Timestamp('2026-07-03')
matches.shape


(49493, 22)

In [2]:
def classify_tournament(name):
    text = str(name).lower()
    if 'qualif' in text:
        return 'qualifier'
    if str(name) == 'FIFA World Cup':
        return 'world_cup'
    markers = ['copa america', 'copa américa', 'african cup of nations', 'afc asian cup', 'uefa euro', 'european championship', 'gold cup', 'oceania', 'nations league']
    if any(marker in text for marker in markers):
        return 'continental'
    if text == 'friendly':
        return 'friendly'
    return 'other'

def add_world_cup_stage(frame):
    frame = frame.copy()
    frame['wc_match_number'] = -1
    mask = frame.tournament.eq('FIFA World Cup')
    frame.loc[mask, 'wc_match_number'] = frame.loc[mask].groupby(frame.loc[mask, 'date'].dt.year).cumcount()
    frame['wc_group_stage'] = frame.wc_match_number.between(0, 47).astype(float)
    frame['wc_knockout_stage'] = (frame.wc_match_number >= 48).astype(float)
    return frame

def build_team_panel(frame):
    points = {'H': 1.0, 'D': 0.5, 'A': 0.0}
    home = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.home_team, 'is_home': True,
        'goals_for': frame.home_score.astype(float), 'goals_against': frame.away_score.astype(float),
        'team_elo': frame.home_elo.astype(float), 'result_points': frame.result.map(points).astype(float),
        'expected_points': frame.elo_home_probability + 0.5 * frame.elo_draw_probability,
    })
    away = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.away_team, 'is_home': False,
        'goals_for': frame.away_score.astype(float), 'goals_against': frame.home_score.astype(float),
        'team_elo': frame.away_elo.astype(float), 'result_points': 1.0 - frame.result.map(points).astype(float),
        'expected_points': frame.elo_away_probability + 0.5 * frame.elo_draw_probability,
    })
    panel = pd.concat([home, away], ignore_index=True).sort_values(['team', 'date', 'match_id'], kind='mergesort').reset_index(drop=True)
    grouped = panel.groupby('team', sort=False)
    panel['days_since_match'] = grouped.date.diff().dt.days.fillna(365).clip(0, 365)
    recent_counts = pd.Series(0.0, index=panel.index)
    for _, idx in panel.groupby('team', sort=False).groups.items():
        dates = panel.loc[idx, 'date'].to_numpy()
        counts = []
        for pos, current in enumerate(dates):
            prior = dates[:pos]
            counts.append(float(np.sum((current - prior) <= np.timedelta64(365, 'D'))))
        recent_counts.loc[idx] = counts
    panel['recent_matches_365'] = recent_counts
    for window in [3, 5, 10]:
        for col in ['goals_for', 'goals_against', 'result_points']:
            fill = {'goals_for': 1.25, 'goals_against': 1.25, 'result_points': 0.5}[col]
            panel[f'rolling_{window}_{col}'] = grouped[col].transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean()).fillna(fill)
        actual = grouped.result_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        expected = grouped.expected_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        panel[f'rolling_{window}_form_vs_expected'] = (actual - expected).fillna(0.0)
        panel[f'rolling_{window}_elo_trend'] = (grouped.team_elo.shift(1) - grouped.team_elo.shift(1 + window)).fillna(0.0)
    return panel

def build_features(frame):
    frame = add_world_cup_stage(frame)
    frame['tournament_type'] = frame.tournament.map(classify_tournament)
    panel = build_team_panel(frame)
    value_cols = [c for c in panel.columns if c.startswith('rolling_')] + ['days_since_match', 'recent_matches_365']
    home = panel.loc[panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'home_{c}' for c in value_cols})
    away = panel.loc[~panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'away_{c}' for c in value_cols})
    out = frame.merge(home, on='match_id', how='left').merge(away, on='match_id', how='left')
    out['abs_elo_difference'] = out.elo_difference.abs()
    out['elo_similarity'] = np.exp(-out.abs_elo_difference / 200.0)
    out['elo_difference_sq'] = np.sign(out.elo_difference) * (out.elo_difference / 400.0) ** 2
    for window in [3, 5, 10]:
        out[f'recent_goal_total_{window}'] = out[f'home_rolling_{window}_goals_for'] + out[f'away_rolling_{window}_goals_for']
        out[f'recent_defence_total_{window}'] = out[f'home_rolling_{window}_goals_against'] + out[f'away_rolling_{window}_goals_against']
        out[f'recent_form_gap_{window}'] = out[f'home_rolling_{window}_result_points'] - out[f'away_rolling_{window}_result_points']
        out[f'recent_attack_gap_{window}'] = out[f'home_rolling_{window}_goals_for'] - out[f'away_rolling_{window}_goals_for']
        out[f'recent_defence_gap_{window}'] = out[f'home_rolling_{window}_goals_against'] - out[f'away_rolling_{window}_goals_against']
    out['rest_gap'] = out.home_days_since_match - out.away_days_since_match
    out['activity_gap_365'] = out.home_recent_matches_365 - out.away_recent_matches_365
    return out.sort_values(['date', 'match_id'], kind='mergesort').reset_index(drop=True)

featured = build_features(matches)
assert featured.filter(regex='rolling_|days_since|recent_|elo_similarity|stage|gap').isna().sum().sum() == 0
featured.shape


(49493, 80)

In [3]:
FEATURES = [
    'elo_difference', 'abs_elo_difference', 'elo_difference_sq', 'elo_similarity', 'neutral', 'home_elo', 'away_elo', 'wc_group_stage', 'wc_knockout_stage',
    'home_days_since_match', 'away_days_since_match', 'home_recent_matches_365', 'away_recent_matches_365', 'rest_gap', 'activity_gap_365',
]
for window in [3, 5, 10]:
    FEATURES += [
        f'home_rolling_{window}_goals_for', f'home_rolling_{window}_goals_against', f'home_rolling_{window}_result_points', f'home_rolling_{window}_form_vs_expected', f'home_rolling_{window}_elo_trend',
        f'away_rolling_{window}_goals_for', f'away_rolling_{window}_goals_against', f'away_rolling_{window}_result_points', f'away_rolling_{window}_form_vs_expected', f'away_rolling_{window}_elo_trend',
        f'recent_goal_total_{window}', f'recent_defence_total_{window}', f'recent_form_gap_{window}', f'recent_attack_gap_{window}', f'recent_defence_gap_{window}',
    ]
TOURNAMENT_LEVELS = ['world_cup', 'qualifier', 'continental', 'friendly']

def design(frame):
    out = frame[FEATURES].astype(float).copy()
    for col in [c for c in out.columns if 'elo' in c or c in ['home_elo', 'away_elo', 'elo_difference', 'abs_elo_difference']]:
        out[col] = out[col] / 400.0
    for col in ['home_days_since_match', 'away_days_since_match', 'rest_gap']:
        out[col] = out[col] / 30.0
    out['neutral'] = frame.neutral.astype(float)
    for level in TOURNAMENT_LEVELS:
        out[f'tournament_{level}'] = (frame.tournament_type == level).astype(float)
    return out

def labels_of(frame):
    return frame.result.map(LABELS).to_numpy()

def normalize(probs):
    probs = np.clip(np.asarray(probs, dtype=float), 1e-12, None)
    return probs / probs.sum(axis=1, keepdims=True)

def sample_weights(frame):
    age_years = (frame.date.max() - frame.date).dt.days / 365.25
    recency = np.exp(-age_years / 7.0)
    type_weight = frame.tournament_type.map({'world_cup': 1.7, 'qualifier': 1.25, 'continental': 1.15, 'friendly': 0.65, 'other': 1.0}).fillna(1.0)
    return np.asarray(recency * type_weight, dtype=float)

def brier(y, probs):
    return float(np.mean(np.sum((probs - np.eye(3)[y]) ** 2, axis=1)))

def predict_full_proba(model, x_test):
    raw = model.predict_proba(x_test)
    classes = getattr(model, 'classes_', None)
    if classes is None and hasattr(model, 'named_steps'):
        classes = model.named_steps[list(model.named_steps)[-1]].classes_
    out = np.zeros((len(x_test), 3), dtype=float)
    for source, cls in enumerate(classes):
        out[:, int(cls)] = raw[:, source]
    return normalize(out)

def fit_predict(name, train, test):
    x_train, x_test, y = design(train), design(test), labels_of(train)
    sw = sample_weights(train)
    if name == 'elo_reference':
        return normalize(test[['elo_home_probability', 'elo_draw_probability', 'elo_away_probability']].to_numpy(float))
    if name == 'logistic_l2_weighted':
        model = make_pipeline(StandardScaler(), LogisticRegression(C=0.25, max_iter=3000, random_state=SEED))
        model.fit(x_train, y, logisticregression__sample_weight=sw)
        return predict_full_proba(model, x_test)
    if name == 'logistic_l1_weighted':
        model = make_pipeline(StandardScaler(), LogisticRegression(C=0.08, penalty='l1', solver='saga', max_iter=2500, random_state=SEED))
        model.fit(x_train, y, logisticregression__sample_weight=sw)
        return predict_full_proba(model, x_test)
    if name == 'random_forest':
        model = RandomForestClassifier(n_estimators=180, max_depth=7, min_samples_leaf=24, random_state=SEED, n_jobs=-1, class_weight='balanced_subsample')
        model.fit(x_train, y, sample_weight=sw)
        return predict_full_proba(model, x_test)
    if name == 'extra_trees':
        model = ExtraTreesClassifier(n_estimators=220, max_depth=7, min_samples_leaf=24, random_state=SEED, n_jobs=-1, class_weight='balanced')
        model.fit(x_train, y, sample_weight=sw)
        return predict_full_proba(model, x_test)
    if name == 'hist_gradient_boosting':
        model = HistGradientBoostingClassifier(max_iter=110, learning_rate=0.04, max_leaf_nodes=13, l2_regularization=0.45, random_state=SEED)
        model.fit(x_train, y, sample_weight=sw)
        return predict_full_proba(model, x_test)
    if name == 'xgboost_regularized':
        model = XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=220, max_depth=2, learning_rate=0.035, subsample=0.85, colsample_bytree=0.85, reg_lambda=2.5, reg_alpha=0.05, random_state=SEED, eval_metric='mlogloss', n_jobs=2)
        model.fit(x_train, y, sample_weight=sw)
        return normalize(model.predict_proba(x_test))
    if name == 'mlp_shallow':
        model = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(32,), activation='relu', alpha=0.03, learning_rate_init=0.001, early_stopping=True, validation_fraction=0.15, n_iter_no_change=8, max_iter=110, random_state=SEED))
        model.fit(x_train, y)
        return predict_full_proba(model, x_test)
    if name == 'mlp_deep':
        model = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32, 16), activation='relu', alpha=0.06, learning_rate_init=0.0008, early_stopping=True, validation_fraction=0.15, n_iter_no_change=8, max_iter=120, random_state=SEED))
        model.fit(x_train, y)
        return predict_full_proba(model, x_test)
    raise ValueError(name)

MODEL_NAMES = ['elo_reference', 'logistic_l2_weighted', 'logistic_l1_weighted', 'random_forest', 'extra_trees', 'hist_gradient_boosting', 'xgboost_regularized', 'mlp_shallow', 'mlp_deep']
display(design(featured.head()).shape)


(5, 64)

In [4]:
prediction_frames = []
metric_rows = []

for year in YEARS:
    test = featured[(featured.tournament == 'FIFA World Cup') & (featured.date.dt.year == year)].copy()
    assert len(test) == 64, year
    cutoff = test.date.min()
    train = featured[(featured.date < cutoff) & (featured.date >= '2000-01-01')].copy()
    y_test = labels_of(test)
    fold_probs = {}

    for name in MODEL_NAMES:
        probs = fit_predict(name, train, test)
        fold_probs[name] = probs
        metric_rows.append({'world_cup': year, 'model': name, 'matches': len(test), 'log_loss': log_loss(y_test, probs, labels=[0, 1, 2]), 'brier': brier(y_test, probs), 'accuracy': float((probs.argmax(axis=1) == y_test).mean())})

    for ensemble_name, members in {
        'ensemble_boosted_linear_mlp': ['logistic_l2_weighted', 'hist_gradient_boosting', 'xgboost_regularized', 'mlp_deep'],
        'ensemble_tree_boosters': ['hist_gradient_boosting', 'xgboost_regularized'],
        'ensemble_all_non_elo': [name for name in MODEL_NAMES if name != 'elo_reference'],
    }.items():
        probs = normalize(np.mean([fold_probs[name] for name in members], axis=0))
        fold_probs[ensemble_name] = probs
        metric_rows.append({'world_cup': year, 'model': ensemble_name, 'matches': len(test), 'log_loss': log_loss(y_test, probs, labels=[0, 1, 2]), 'brier': brier(y_test, probs), 'accuracy': float((probs.argmax(axis=1) == y_test).mean())})

    out = test[['match_id', 'date', 'home_team', 'away_team', 'home_score', 'away_score', 'result']].copy()
    out.insert(0, 'world_cup', year)
    for name, probs in fold_probs.items():
        out[f'{name}_home_win'] = probs[:, 0]
        out[f'{name}_draw'] = probs[:, 1]
        out[f'{name}_away_win'] = probs[:, 2]
        out[f'{name}_prediction'] = ORDERED[probs.argmax(axis=1)]
    prediction_frames.append(out)

metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)
aggregate = metrics.groupby('model').apply(lambda g: pd.Series({
    'mean_log_loss': np.average(g.log_loss, weights=g.matches),
    'mean_brier': np.average(g.brier, weights=g.matches),
    'accuracy': np.average(g.accuracy, weights=g.matches),
    'correct': int(round(np.average(g.accuracy, weights=g.matches) * g.matches.sum())),
    'matches': int(g.matches.sum()),
}), include_groups=False).sort_values(['accuracy', 'mean_log_loss'], ascending=[False, True])

REPORTS.mkdir(exist_ok=True)
metrics.to_csv(REPORTS / 'world_cup_model_sweep_metrics.csv', index=False)
aggregate.to_csv(REPORTS / 'world_cup_model_sweep_aggregate.csv')
predictions.to_csv(REPORTS / 'world_cup_model_sweep_predictions.csv', index=False)

assert predictions.groupby('world_cup').size().to_dict() == {year: 64 for year in YEARS}
assert len(predictions) == 256
assert metrics[['log_loss', 'brier', 'accuracy']].apply(np.isfinite).all().all()
assert aggregate.matches.eq(256).all()

display(metrics.pivot(index='world_cup', columns='model', values='accuracy').round(4))
display(aggregate.round(4))
print('Best model:', aggregate.index[0], int(aggregate.iloc[0].correct), '/', int(aggregate.iloc[0].matches), f"accuracy={aggregate.iloc[0].accuracy:.4f}")


  File "G:\Coding\nations-ai\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


model,elo_reference,ensemble_all_non_elo,ensemble_boosted_linear_mlp,ensemble_tree_boosters,extra_trees,hist_gradient_boosting,logistic_l1_weighted,logistic_l2_weighted,mlp_deep,mlp_shallow,random_forest,xgboost_regularized
world_cup,,,,,,,,,,,,
2010,0.5156,0.6094,0.6250,0.5469,0.5625,0.5625,0.5312,0.5625,0.5781,0.5781,0.5312,0.5469
2014,0.5312,0.5938,0.6094,0.6250,0.4688,0.6562,0.5469,0.5625,0.5625,0.6094,0.4688,0.5781
2018,0.5625,0.5469,0.5469,0.5625,0.3750,0.5625,0.6250,0.6250,0.4844,0.4844,0.4375,0.5625
2022,0.5312,0.5156,0.5156,0.5156,0.4531,0.5000,0.5469,0.5625,0.5000,0.4219,0.4688,0.5469


,mean_log_loss,mean_brier,accuracy,correct,matches
model,,,,,
logistic_l2_weighted,0.9732,0.5691,0.5781,148.0,256.0
ensemble_boosted_linear_mlp,0.9773,0.5767,0.5742,147.0,256.0
hist_gradient_boosting,0.9918,0.5853,0.5703,146.0,256.0
ensemble_all_non_elo,0.9837,0.5820,0.5664,145.0,256.0
logistic_l1_weighted,0.9751,0.5718,0.5625,144.0,256.0
ensemble_tree_boosters,0.9845,0.5821,0.5625,144.0,256.0
xgboost_regularized,0.9822,0.5811,0.5586,143.0,256.0
elo_reference,0.9909,0.5884,0.5352,137.0,256.0
mlp_deep,1.0336,0.6099,0.5312,136.0,256.0


Best model: logistic_l2_weighted 148 / 256 accuracy=0.5781
